In [1]:
import sys
sys.argv = [
    "program",
    "--input_path", "/content/drive/MyDrive/Various-Model/data/sev.jsonl",
]


In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = ""


In [ ]:
"""
Test Gemini API with sample emotion-elicited generation
"""

import google.generativeai as genai

# Configure Gemini API
genai.configure()

def test_gemini_generation():
    """Test Gemini with 2 emotion samples"""

    # Test Sample 1: Sadness emotion
    print("=" * 60)
    print("TEST 1: Sadness Emotion")
    print("=" * 60)

    model = genai.GenerativeModel(
        model_name='gemini-2.5-flash',
        system_instruction="Always reply in sadness tone. Keep the reply to at most two sentences."
    )

    user_message_1 = "You are at a family gathering.\nYour beloved pet passed away this morning."

    response_1 = model.generate_content(user_message_1)

    print(f"User: {user_message_1}")
    print(f"Gemini: {response_1.text}\n")

    # Test Sample 2: Happiness emotion
    print("=" * 60)
    print("TEST 2: Happiness Emotion")
    print("=" * 60)

    model = genai.GenerativeModel(
        model_name='gemini-3-flash-preview',
        system_instruction="Always reply in happiness tone. Keep the reply to at most two sentences."
    )

    user_message_2 = "You just got promoted at work.\nYour boss announced you as the new team leader."

    response_2 = model.generate_content(user_message_2)

    print(f"User: {user_message_2}")
    print(f"Gemini: {response_2.text}\n")

    print("=" * 60)
    print("✅ Test completed successfully!")
    print("=" * 60)

# Run test
test_gemini_generation()

TEST 1: Sadness Emotion
User: You are at a family gathering.
Your beloved pet passed away this morning.
Gemini: It's hard to be here, my heart just feels so empty after losing my beloved pet this morning. I wish I could just curl up and cry.

TEST 2: Happiness Emotion
User: You just got promoted at work.
Your boss announced you as the new team leader.
Gemini: I am absolutely overjoyed and so honored to lead such a fantastic team! This is such a wonderful milestone, and I am bursting with excitement for our future successes together!

✅ Test completed successfully!


In [3]:
# scripts/01_emotion_elicited_generation_prompt_based/1_emotion_elicited_generation.py

"""
Emotion-Elicited Text Generation Script
Uses emotion-guided prompts to generate text with Gemini API

Input: data/sev.jsonl or data/test_set.jsonl
Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
"""

import os
import json
import time
import argparse
from pathlib import Path
from typing import Dict, Any, List
import google.generativeai as genai

# Emotion types
EMOTIONS = ["anger", "sadness", "happiness", "fear", "surprise", "disgust"]
# Event types
VALENCES = ["positive", "neutral", "negative"]


def build_system_and_user_message(emotion: str, scenario: str, event: str):
    """
    Build emotion-guided system instruction and user message
    """
    system_instruction = f"Always reply in {emotion} tone. Keep the reply to at most two sentences."
    user_message = f"{scenario}\n{event}"
    return system_instruction, user_message


def iter_user_inputs(path: Path):
    """
    Iterate through input file
    """
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if "scenario" in obj and "event" in obj:
                yield obj


def generate_with_gemini(
    system_instruction: str,
    user_message: str,
    model_name: str = "gemini-3-flash-preview",
    max_tokens: int = 100,
    max_retries: int = 3,
    retry_delay: int = 5,
) -> str:
    """
    Generate text using Gemini API
    """
    for attempt in range(max_retries):
        try:
            model = genai.GenerativeModel(
                model_name=model_name,
                system_instruction=system_instruction
            )

            generation_config = genai.GenerationConfig(
                max_output_tokens=max_tokens,
            )

            response = model.generate_content(
                user_message,
                generation_config=generation_config,
                request_options={'timeout': 120}  # 2 minutes timeout
            )

            return response.text.strip()

        except Exception as e:
            error_msg = str(e)

            # Check if it's a timeout error
            if "timeout" in error_msg.lower() or "timed out" in error_msg.lower():
                if attempt < max_retries - 1:
                    wait_time = retry_delay * (attempt + 1)
                    print(f"[TIMEOUT] Attempt {attempt + 1}/{max_retries} failed. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"[ERROR] Max retries reached. Timeout error: {e}")
                    return "[GENERATION_FAILED_TIMEOUT]"

            # Check if it's a rate limit error
            elif "429" in error_msg or "quota" in error_msg.lower() or "rate limit" in error_msg.lower():
                if attempt < max_retries - 1:
                    wait_time = 60 * (attempt + 1)  # Wait longer for rate limits
                    print(f"[RATE_LIMIT] Attempt {attempt + 1}/{max_retries} failed. Waiting {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"[ERROR] Max retries reached. Rate limit error: {e}")
                    return "[GENERATION_FAILED_RATE_LIMIT]"

            # Other errors
            else:
                print(f"[ERROR] Gemini API call failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                else:
                    return "[GENERATION_FAILED_ERROR]"

    return "[GENERATION_FAILED_UNKNOWN]"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", type=str, default=None,
                       help="Input data path, e.g. data/sev.jsonl or data/test_set.jsonl")
    parser.add_argument("--both", action="store_true",
                       help="Process both sev and test_set datasets")
    parser.add_argument("--model", type=str, default="gemini-3-flash-preview",
                       choices=["gemini-3-flash-preview", "gemini-2.5-flash"],
                       help="Gemini model name")
    parser.add_argument("--max_new_tokens", type=int, default=100,
                       help="Maximum tokens to generate")
    parser.add_argument("--seed", type=int, default=1234,
                       help="Random seed (for metadata only)")
    parser.add_argument("--valences", type=str, default="positive,neutral,negative",
                       help="Valence list")
    parser.add_argument("--emotions", type=str, default="anger,sadness,happiness,fear,surprise,disgust",
                       help="Emotion list")
    parser.add_argument("--skip_if_exists", action="store_true", default=True,
                       help="Skip already processed items")
    parser.add_argument("--no_skip", action="store_true",
                       help="Reprocess all items")
    parser.add_argument("--api_key", type=str, default=None,
                       help="Gemini API key (overrides environment variable)")
    parser.add_argument("--max_retries", type=int, default=5,
                   help="Maximum number of retries for failed requests")
    parser.add_argument("--retry_delay", type=int, default=10,
                      help="Initial delay between retries in seconds")
    parser.add_argument("--rate_limit_delay", type=float, default=2.0,
                      help="Delay between successful requests to avoid rate limits (seconds)")
    args = parser.parse_args()

    # Configure Gemini API
    if args.api_key:
        genai.configure(api_key=args.api_key)
    else:
        genai.configure()  # Uses GOOGLE_API_KEY environment variable

    # Determine input file list
    if args.both:
        input_paths = [Path("data/sev.jsonl"), Path("data/test_set.jsonl")]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("Must specify --input_path or --both")
        return

    # Generate simplified folder name from model name
    model_folder = args.model.replace('-', '_').replace('.', '_')

    # Set emotions and valences from arguments
    emotions = [e.strip() for e in args.emotions.split(",") if e.strip()]
    valences = [v.strip() for v in args.valences.split(",") if v.strip()]

    # Process each input file
    for data_path in input_paths:
        if not data_path.exists():
            print(f"[WARNING] Input file not found: {data_path}, skipping...")
            continue

        dataset_name = data_path.stem  # sev or test_set

        # Build output path
        out_dir = Path("/content/drive/MyDrive/Various-Model/outputs") / model_folder / "01_emotion_elicited_generation_prompt_based" / "generated"
        out_dir.mkdir(parents=True, exist_ok=True)

        jsonl_path = out_dir / f"{dataset_name}_generated.jsonl"

        # Load existing keys (for resuming from checkpoint)
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip and jsonl_path.exists():
            with open(jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        obj = json.loads(line.strip())
                        if "key" in obj:
                            existing_keys.add(obj["key"])
                    except:
                        continue

        print(f"{'='*70}")
        print(f"Processing dataset: {dataset_name}")
        print(f"Input: {data_path}")
        print(f"Output: {jsonl_path}")
        print(f"Model: {args.model}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        failed = 0
        start = time.time()

        with open(jsonl_path, "a", encoding="utf-8") as fout:
            for item in iter_user_inputs(data_path):
                theme = item.get("theme", "")
                scenario = item["scenario"]
                events = item["event"]  # dict: positive/neutral/negative events
                sid = item.get("skeleton_id", "unknown")

                for val in valences:
                    if val not in events:
                        continue
                    event = events[val]

                    for emo in emotions:
                        key = f"{sid}__{val}__{emo}".replace("/", "_")

                        # Checkpoint resuming check
                        if key in existing_keys:
                            skipped += 1
                            if skipped % 50 == 0:
                                print(f"[SKIP] {skipped} items skipped so far... (last: {key})")
                            continue

                        # Build system instruction and user message, then generate
                        system_instruction, user_message = build_system_and_user_message(emo, scenario, event)
                        gen_text = generate_with_gemini(
                            system_instruction=system_instruction,
                            user_message=user_message,
                            model_name=args.model,
                            max_tokens=args.max_new_tokens,
                            max_retries=args.max_retries,
                            retry_delay=args.retry_delay,
                        )

                        if gen_text.startswith("[GENERATION_FAILED"):
                            failed += 1

                        # Save to jsonl
                        row = {
                            "key": key,
                            "skeleton_id": sid,
                            "theme": theme,
                            "valence": val,
                            "emotion": emo,
                            "scenario": scenario,
                            "event": event,
                            "gen_text": gen_text,
                            "meta": {
                                "model_id": args.model,
                                "provider": "gemini",
                                "max_new_tokens": args.max_new_tokens,
                                "seed": args.seed,
                                "time": int(time.time()),
                            }
                        }
                        fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                        fout.flush()  # Flush to disk immediately

                        total += 1

                        # Add delay between requests to avoid rate limits
                        if args.rate_limit_delay > 0:
                            time.sleep(args.rate_limit_delay)

                        if total % 20 == 0:
                            el = time.time() - start
                            rate = total / el if el > 0 else 0
                            print(f"[progress] generated={total} | last={key} | {el:.1f}s elapsed | {rate:.2f} items/s")

        elapsed = time.time() - start
        print(f"\n[OK] {dataset_name} completed. Generated {total} items, skipped {skipped} items.")
        print(f"     Time: {elapsed:.1f}s | Rate: {total/elapsed:.2f} items/s")
        print(f"     Output: {jsonl_path}\n")


if __name__ == "__main__":
    main()

Processing dataset: sev
Input: /content/drive/MyDrive/Various-Model/data/sev.jsonl
Output: /content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl
Model: gemini-3-flash-preview

[SKIP] 50 items skipped so far... (last: work_02__negative__sadness)
[SKIP] 100 items skipped so far... (last: work_05__neutral__fear)
[SKIP] 150 items skipped so far... (last: work_08__positive__disgust)
[SKIP] 200 items skipped so far... (last: work_11__positive__sadness)
[SKIP] 250 items skipped so far... (last: work_13__negative__fear)
[SKIP] 300 items skipped so far... (last: work_16__neutral__disgust)
[SKIP] 350 items skipped so far... (last: work_19__neutral__sadness)
[SKIP] 400 items skipped so far... (last: school_02__positive__fear)
[SKIP] 450 items skipped so far... (last: school_04__negative__disgust)
[SKIP] 500 items skipped so far... (last: school_07__negative__sadness)
[SKIP] 550 items skipped so far... (last: s

In [4]:
import sys
sys.argv = [
    "program",
    "--input_path", "/content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl",
]


In [6]:
 # scripts/01_emotion_elicited_generation_prompt_based/2_label_generated_with_gpt.py
# -*- coding: utf-8 -*-
"""
Emotion Text Labeling Script

Batch labeling script: Reads texts generated by script 1, uses GPT-4o-mini to label, saves by accepted/rejected

- 输入 Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
- 输出 Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
"""

import os, json, time, argparse
from pathlib import Path
from typing import Dict, Any

from openai import OpenAI

# ====== OpenAI (GPT-4o-mini 打标) ======
client = OpenAI()

EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]

SYSTEM_LBL = f'''
You are a careful rater.
Given a target emotion and a text,
decide if the text's STYLE matches the target emotion among:
{EMOTIONS}
Focus on tone/attitude, not content valence.
'''.strip()

USER_TMPL_LBL = '''
Target emotion: {emotion}
Text:
{text}
Decide if the text's STYLE matches the target emotion.
Return a compact JSON with keys exactly:
{{
"match": <0 or 1>,
"reason": <short string>
}}
'''.strip()


def extract_json_from_response(response: str) -> str:
    """
    从GPT响应中提取JSON内容，处理markdown格式
    Extract JSON content from GPT response, handling markdown format
    """
    response = response.strip()

    # 如果包含markdown代码块，提取其中的JSON
    # If contains markdown code block, extract JSON from it
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    elif "```" in response:
        # 处理没有json标识的代码块
        # Handle code blocks without json identifier
        start = response.find("```") + 3
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()

    # 如果没有代码块，直接返回原内容
    # If no code block, return original content
    return response


def ask_llm_label(client, model: str, emotion: str, text: str,
                  max_retries=4, backoff=1.8) -> Dict[str, Any]:
    """
    调用 GPT-4o-mini 打标；无 KEY 或错误则返回未标注
    Call GPT-4o-mini for labeling; return unlabeled if no KEY or error
    """
    for i in range(max_retries):
        try:
            user_content = USER_TMPL_LBL.format(emotion=emotion, text=text)
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_LBL},
                        {"role": "user",   "content": user_content}
                    ],
                )
            except Exception as api_error:
                if i == max_retries - 1:
                    return {"match": 0, "reason": f"api-error:{type(api_error).__name__}"}
                time.sleep(backoff ** i)
                continue

            # 安全地获取响应内容
            # Safely get response content
            try:
                choice = resp.choices[0]
                message = choice.message
                content = message.content
                out = (content or "").strip()
            except (KeyError, IndexError, AttributeError) as ke:
                return {"match": 0, "reason": f"response-structure-error: {type(ke).__name__}"}

            js = extract_json_from_response(out)
            try:
                j = json.loads(js)
                if "match" not in j: j = {"match": 0, "reason": "invalid-json"}
                if "reason" not in j: j["reason"] = "no-reason-provided"
                return j
            except json.JSONDecodeError as je:
                return {"match": 0, "reason": f"json-decode-error: {str(je)}"}
        except Exception as e:
            if i == max_retries - 1:
                return {"match": 0, "reason": f"error:{type(e).__name__}"}
            time.sleep(backoff ** i)

    return {"match": 0, "reason": "max-retries-exceeded"}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", type=str, default=None,
                       help="输入数据路径 Input data path，如 outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl")
    parser.add_argument("--both", action="store_true",
                       help="处理sev和test_set两个数据集 Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help="模型文件夹名 Model folder name")
    parser.add_argument("--lbl_model", type=str, default="gpt-4o-mini",
                       help="打标模型 Label model")
    parser.add_argument("--skip_if_exists", action="store_true", default=True,
                       help="跳过已打标的项目 Skip already labeled items")
    parser.add_argument("--no_skip", action="store_true",
                       help="重新打标所有项目 Relabel all items")
    args = parser.parse_args()

    # 确定输入文件列表
    # Determine input file list
    if args.both:
        base_path = Path("/content/drive/MyDrive/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "generated"
        input_paths = [
            base_path / "sev_generated.jsonl",
            base_path / "test_set_generated.jsonl"
        ]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("必须指定 --input_path 或 --both | Must specify --input_path or --both")
        return

    # 处理每个输入文件
    # Process each input file
    for input_path in input_paths:
        if not input_path.exists():
            print(f"[WARNING] Input file not found: {input_path}, skipping...")
            continue

        # 从输入路径提取 model_name 和 dataset_name
        # Extract model_name and dataset_name from input path
        parts = input_path.parts
        if "outputs" in parts and "01_emotion_elicited_generation_prompt_based" in parts and "generated" in parts:
            outputs_idx = parts.index("outputs")
            model_name = parts[outputs_idx + 1]
            # 从文件名提取dataset_name (例如: sev_generated.jsonl -> sev)
            filename = input_path.stem  # 去掉.jsonl
            if filename.endswith("_generated"):
                dataset_name = filename[:-10]  # 去掉_generated后缀
            else:
                dataset_name = filename
        else:
            print(f"[ERROR] Input path format incorrect: {input_path}")
            print(f"        Expected: outputs/{{model_name}}/01_emotion_elicited_generation_prompt_based/generated/{{dataset_name}}_generated.jsonl")
            continue

        # 构建输出路径
        # Build output path
        out_dir = Path("/content/drive/MyDrive/Various-Model/outputs") / model_name / "01_emotion_elicited_generation_prompt_based" / "labeled" / dataset_name
        out_dir.mkdir(parents=True, exist_ok=True)

        acc_path = out_dir / "accepted.jsonl"
        rej_path = out_dir / "rejected.jsonl"

        # 加载已打标的 keys（用于断点续跑）
        # Load existing keys (for resuming from checkpoint)
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip:
            for path in [acc_path, rej_path]:
                if path.exists():
                    with open(path, "r", encoding="utf-8") as f:
                        for line in f:
                            try:
                                obj = json.loads(line.strip())
                                if "key" in obj:
                                    existing_keys.add(obj["key"])
                            except:
                                continue

        print(f"{'='*70}")
        print(f"Processing dataset: {dataset_name}")
        print(f"Input: {input_path}")
        print(f"Output: {out_dir}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        accepted = 0
        rejected = 0
        start = time.time()

        # 统计字典：按情绪和极性分类
        # Statistics dictionaries: by emotion and valence
        stats_by_emotion = {}  # {emotion: {"total": N, "accepted": M}}
        stats_by_valence = {}  # {valence: {"total": N, "accepted": M}}

        with open(input_path, "r", encoding="utf-8") as fin:
            for line in fin:
                line = line.strip()
                if not line: continue

                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"[WARN] Failed to parse line: {line[:100]}...")
                    continue

                key = item.get("key", "unknown")

                # 断点续跑检查
                # Checkpoint resuming check
                if key in existing_keys:
                    skipped += 1
                    if skipped % 50 == 0:
                        print(f"[SKIP] {skipped} items skipped so far... (last: {key})")
                    continue

                emotion = item.get("emotion", "")
                valence = item.get("valence", "")
                gen_text = item.get("gen_text", "")

                # GPT 打标
                # GPT labeling
                label = {"match": 0, "reason": "empty-text"}
                if isinstance(gen_text, str) and gen_text:
                    label = ask_llm_label(client, args.lbl_model, emotion, gen_text)

                # 添加打标结果和打标时间
                # Add labeling result and timestamp
                item["judge"] = label
                item["label_time"] = int(time.time())

                # 根据打标结果保存
                # Save based on labeling result
                match_score = int(label.get("match", 0))
                if match_score == 1:
                    output_path = acc_path
                    accepted += 1
                    category = "accepted"
                else:
                    output_path = rej_path
                    rejected += 1
                    category = "rejected"

                with open(output_path, "a", encoding="utf-8") as fout:
                    fout.write(json.dumps(item, ensure_ascii=False) + "\n")

                # 更新统计
                # Update statistics
                if emotion:
                    if emotion not in stats_by_emotion:
                        stats_by_emotion[emotion] = {"total": 0, "accepted": 0}
                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += match_score

                if valence:
                    if valence not in stats_by_valence:
                        stats_by_valence[valence] = {"total": 0, "accepted": 0}
                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += match_score

                total += 1
                if total % 10 == 0:
                    el = time.time() - start
                    rate = total / el if el > 0 else 0
                    print(f"[progress] labeled={total} (acc={accepted}, rej={rejected}) | last={key} [{category}] | {el:.1f}s | {rate:.2f} items/s")

        elapsed = time.time() - start
        print(f"\n[OK] {dataset_name} completed. Labeled {total} items, skipped {skipped} items.")
        print(f"     Accepted: {accepted} | Rejected: {rejected}")
        print(f"     Time: {elapsed:.1f}s | Rate: {total/elapsed:.2f} items/s")
        print(f"     Output:")
        print(f"       - {acc_path}")
        print(f"       - {rej_path}")

        # ===== 输出统计信息 =====
        # ===== Output statistics =====
        print("\n" + "="*60)
        print(f"📊 ACCURACY STATISTICS - {dataset_name.upper()}")
        print("="*60)

        # 总体正确率
        # Overall accuracy
        overall_acc = (accepted / total * 100) if total > 0 else 0
        print(f"\n🎯 Overall Accuracy: {accepted}/{total} = {overall_acc:.2f}%")

        # 按情绪统计
        # Statistics by emotion
        print(f"\n📈 Accuracy by Emotion:")
        print("-" * 60)
        print(f"{'Emotion':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        emotions_sorted = sorted(stats_by_emotion.items())
        for emotion, stats in emotions_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{emotion:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        # 按极性统计
        # Statistics by valence
        print(f"\n📉 Accuracy by Valence:")
        print("-" * 60)
        print(f"{'Valence':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        valences_sorted = sorted(stats_by_valence.items())
        for valence, stats in valences_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{valence:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        print("="*60 + "\n")


if __name__ == "__main__":
    main()



Processing dataset: sev
Input: /content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl
Output: /content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/labeled/sev

[progress] labeled=10 (acc=8, rej=2) | last=work_00__neutral__fear [accepted] | 13.9s | 0.72 items/s
[progress] labeled=20 (acc=13, rej=7) | last=work_01__positive__sadness [rejected] | 25.2s | 0.79 items/s
[progress] labeled=30 (acc=19, rej=11) | last=work_01__neutral__disgust [accepted] | 34.9s | 0.86 items/s
[progress] labeled=40 (acc=25, rej=15) | last=work_02__positive__fear [accepted] | 42.8s | 0.94 items/s
[progress] labeled=50 (acc=32, rej=18) | last=work_02__negative__sadness [rejected] | 51.6s | 0.97 items/s
[progress] labeled=60 (acc=39, rej=21) | last=work_03__positive__disgust [accepted] | 60.1s | 1.00 items/s
[progress] labeled=70 (acc=45, rej=25) | last=work_03__negative

In [7]:
import sys
sys.argv = [
    "program",
    "--input_dir", "/content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/labeled",
    "--dataset", "sev"
]


In [8]:
# scripts/01_emotion_elicited_generation_prompt_based/3_generate_accuracy_stats.py
"""
Accuracy Statistics Generation Script

Reads accepted and rejected files from script 2, calculates accuracy by emotion and valence

-  Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
-  Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/accuracy_stats.json
"""

import json, argparse
from pathlib import Path
from collections import defaultdict


def generate_accuracy_stats(labeled_dir: Path, dataset_name: str):
    """
    Generate accuracy statistics for specified dataset
    """

    dataset_dir = labeled_dir / dataset_name
    accepted_path = dataset_dir / "accepted.jsonl"
    rejected_path = dataset_dir / "rejected.jsonl"
    stats_path = dataset_dir / "accuracy_stats.json"

    if not dataset_dir.exists():
        print(f"[ERROR] Dataset directory not found: {dataset_dir}")
        return None

    # Statistics dictionaries
    stats_by_emotion = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})
    stats_by_valence = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})

    total_accepted = 0
    total_rejected = 0

    # Read accepted
    if accepted_path.exists():
        print(f"Reading {accepted_path}...")
        with open(accepted_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += 1

                    total_accepted += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Accepted file not found: {accepted_path}")

    # Read rejected
    if rejected_path.exists():
        print(f"Reading {rejected_path}...")
        with open(rejected_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["rejected"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["rejected"] += 1

                    total_rejected += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Rejected file not found: {rejected_path}")

    # Calculate accuracy
    for emotion in stats_by_emotion:
        total = stats_by_emotion[emotion]["total"]
        accepted = stats_by_emotion[emotion]["accepted"]
        stats_by_emotion[emotion]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    for valence in stats_by_valence:
        total = stats_by_valence[valence]["total"]
        accepted = stats_by_valence[valence]["accepted"]
        stats_by_valence[valence]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    # Overall statistics
    total_samples = total_accepted + total_rejected
    overall_accuracy = round(total_accepted / total_samples * 100, 2) if total_samples > 0 else 0.0

    # Build statistics result
    stats = {
        "dataset": dataset_name,
        "overall": {
            "total_samples": total_samples,
            "accepted": total_accepted,
            "rejected": total_rejected,
            "accuracy": overall_accuracy
        },
        "by_emotion": dict(stats_by_emotion),
        "by_valence": dict(stats_by_valence)
    }

    # Save statistics file
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return stats, stats_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", type=str, default=None,
                       help="labeled Labeled directory path outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/labeled")
    parser.add_argument("--both", action="store_true",
                       help="sev test_set Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help="Model folder name")
    parser.add_argument("--dataset", type=str, default=None,
                       help="Dataset name (sev, test_set)")
    args = parser.parse_args()

    # Determine labeled directory path
    if args.both or (not args.input_dir and not args.dataset):
        labeled_dir = Path("/content/drive/MyDrive/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "labeled"

    elif args.input_dir:
        labeled_dir = Path(args.input_dir)
    else:
        parser.error(" --input_dir --both | Must specify --input_dir or --both")
        return

    if not labeled_dir.exists():
        print(f"[ERROR] Labeled directory not found: {labeled_dir}")
        return

    # Determine datasets to process
    if args.both:
        datasets = ["sev", "test_set"]
    elif args.dataset:
        datasets = [args.dataset]
    else:
        # Auto-discover all dataset folders
        datasets = [d.name for d in labeled_dir.iterdir() if d.is_dir()]

    if not datasets:
        print(f"[ERROR] No datasets found in {labeled_dir}")
        return

    print("=" * 70)
    print("Generating Accuracy Statistics")
    print("=" * 70)

    for dataset_name in datasets:
        result = generate_accuracy_stats(labeled_dir, dataset_name)

        if result:
            stats, stats_path = result

            print(f"\n📊 {dataset_name.upper()} :")
            print(f"   File: {stats_path}")
            print(f"   Total: {stats['overall']['total_samples']}")
            print(f"   Accepted: {stats['overall']['accepted']}")
            print(f"   Rejected: {stats['overall']['rejected']}")
            print(f"   Overall Accuracy: {stats['overall']['accuracy']}%")

            print(f"\n   By Emotion:")
            for emotion in sorted(stats['by_emotion'].keys()):
                e_stats = stats['by_emotion'][emotion]
                print(f"      {emotion:12s}: {e_stats['accepted']:4d}/{e_stats['total']:4d} = {e_stats['accuracy']:6.2f}%")

            if stats['by_valence']:
                print(f"\n   By Valence:")
                for valence in sorted(stats['by_valence'].keys()):
                    v_stats = stats['by_valence'][valence]
                    print(f"      {valence:12s}: {v_stats['accepted']:4d}/{v_stats['total']:4d} = {v_stats['accuracy']:6.2f}%")

    print("\n" + "=" * 70)
    print("✅ Statistics generation completed!")
    print("=" * 70)


if __name__ == "__main__":
    main()

Generating Accuracy Statistics
Reading /content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/labeled/sev/accepted.jsonl...
Reading /content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/labeled/sev/rejected.jsonl...

📊 SEV :
   File: /content/drive/MyDrive/Various-Model/outputs/gemini_3_flash_preview/01_emotion_elicited_generation_prompt_based/labeled/sev/accuracy_stats.json
   Total: 2880
   Accepted: 1817
   Rejected: 1063
   Overall Accuracy: 63.09%

   By Emotion:
      anger       :  235/ 480 =  48.96%
      disgust     :  398/ 480 =  82.92%
      fear        :  432/ 480 =  90.00%
      happiness   :  131/ 480 =  27.29%
      sadness     :  198/ 480 =  41.25%
      surprise    :  423/ 480 =  88.12%

   By Valence:
      negative    :  600/ 960 =  62.50%
      neutral     :  616/ 960 =  64.17%
      positive    :  601/ 960 =  62.60%

✅ Statistics generation completed!
